In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r'train_df.csv')

In [4]:
df.isna().sum()

interest_area              0
coding_interest_level    255
preferred_domain           0
current_degree             0
field_of_study             0
current_status             0
known_languages           29
frameworks_known          38
job_type_preference        0
job_experience           342
recommended_job_role       0
dtype: int64

In [5]:
import pickle
with open ('model.pkl' , 'rb') as f:
    model = pickle.load(f)
with open ('encoders.pkl' , 'rb') as f:
    encoders = pickle.load(f)

In [6]:
def train_model(model , df):
    X , y = pre_process(df)
    model.fit(X , y)

In [7]:
def pre_process(df):
    df['job_experience'] = df['job_experience'].fillna(0)
    df['coding_interest_level'] = df['coding_interest_level'].fillna("Beginner")
    df = df.dropna()
    df = label_encoding(df)
    df['job_experience'] = df['job_experience'].apply(lambda x :convert_to_int(x))
    df = one_hot_encoding(df)
    df = reduce_attr(df)
    df['recommended_job_role'] = df['recommended_job_role'].apply(lambda x : x.lower().strip())
    X , y = X_y_from_df(df)
    return X , y

In [8]:
def label_encoding(df):
    label_columns = [
    'interest_area',
    'coding_interest_level',
    'preferred_domain',
    'current_degree',
    'current_status',
    'job_type_preference',
    'field_of_study'
    ]

    for col in label_columns:
        df[col] = df[col].apply(lambda x : x.strip().lower())
        le = encoders[col]
        df[col] = le.transform(df[col])
    return df

In [9]:
def convert_to_int(x):
    if isinstance(x, int):
        return x
    elif isinstance(x, str) and x:  # check if it's a non-empty string
        return int(x[0]) if x[0].isdigit() else None #return the 1st char if it is a digit 
    else:
        return None

In [10]:
def to_array_from_string(x):
    return set([elem.strip().lower() for elem in x.split(',') if elem.strip()])



In [11]:
def one_hot_encoding(df):
    #extracts elements from the string and stores as a set
    df['known_languages'] = df['known_languages'].apply(lambda x : to_array_from_string(x))
    df['frameworks_known'] = df["frameworks_known"].apply(lambda x : to_array_from_string(x))
    
    #attributes for known language 
    df['python'] = 0
    df['c'] = 0
    df['c++'] = 0
    df['sql'] = 0
    df['java'] = 0
    df['javascript'] = 0
    df['r'] = 0
    
    #attributes for frameworks
    df['react'] = 0
    df['node'] = 0
    df['flask'] = 0
    df['django'] = 0
    df['tensorflow'] = 0
    df['pytorch'] = 0
    df['scikit-learn'] = 0

    #one hot encoding for all the languages known (manually)

    all_languages = {"python", "javascript", "java", "c++", "c", "r", "sql"}
    for lang in all_languages:
        df[lang] = df['known_languages'].apply(lambda x: 1 if lang in x else 0)
    
    
    #one hot encoding for all the frameworks known(manually)
    all_frameworks = {"react" , "node" , "flask" , "django" , "tensorflow" , "pytorch" , "scikit-learn"}
    for framework in all_frameworks:
        df[framework] = df['frameworks_known'].apply(lambda x: 1 if framework in x else 0)
    return df

In [12]:
def reduce_attr(df):
    #reduction of attributes for precise output
    #combining similar attributes to one
    
    df['python_r'] = df['python'] +df['r']
    df['java_cpp'] = df['java'] + df['c++']
    df['tf_pt_sk'] = df['tensorflow'] + df['pytorch'] + df['scikit-learn']
    df['django_node'] = df['django'] + df['node']
    
    df['python_r'] = df['python_r'].apply(lambda x: x if x < 1 else 1)
    df['java_cpp'] = df['java_cpp'].apply(lambda x: x if x < 1 else 1)
    df['tf_pt_sk'] = df['tf_pt_sk'].apply(lambda x: x if x < 1 else 1)
    df['django_node'] = df['django_node'].apply(lambda x: x if x < 1 else 1)
    return df

In [13]:
def X_y_from_df(df):
    df.dropna()
    drop_columns = ['known_languages',
       'frameworks_known', 'python','c++', 'java',
        'r', 'django', 'node', 'tensorflow',
       'pytorch', 'scikit-learn',
       'recommended_job_role']


    X = df.drop(drop_columns, axis = 1)
    y = df['recommended_job_role']
    y = encoders['recommended_job_role'].transform(y)
    return X , y

In [14]:
def reuturn_target_map():
    target_mapping = {
    "0": "backenddeveloper",
    "1": "backendintern",
    "2": "cybersecurityanalyst",
    "3": "dataanalyst",
    "4": "developer",
    "5": "developerintern",
    "6": "devops",
    "7": "frontenddeveloper",
    "8": "frontendintern",
    "9": "fullstackintern",
    "10": "intern",
    "11": "mlengineer",
    "12": "mlintern",
    "13": "softwareengineer"
    }
    return target_mapping

In [15]:
df.dropna()

,interest_area,coding_interest_level,preferred_domain,current_degree,field_of_study,current_status,known_languages,frameworks_known,job_type_preference,job_experience,recommended_job_role
1,Creativity,Intermediate,Backend,Masters,CS,Job Seeker,"Java, JavaScript, R","PyTorch, TensorFlow",Full-time Job,1+,ml engineer
2,Creativity,Beginner,Full Stack,Diploma,CS,Job Seeker,Python,"Flask, Node, PyTorch",Onsite,1+,ml engineer
5,Creativity,Advanced,Cybersecurity,Masters,Non-CS,Job Seeker,"C, SQL","None, React, TensorFlow",Full-time Job,1+,ml engineer
7,Logic,Intermediate,AI/ML,Masters,Non-CS,Pursuing Degree,JavaScript,"Flask, Node, React",Full-time Job,3+,frontend developer
8,Creativity,Advanced,Cybersecurity,Bachelors,CS,Intern,"Python, R, SQL","Node, React",Freelance,1+,data analyst
...,...,...,...,...,...,...,...,...,...,...,...
985,Both,Advanced,Cybersecurity,Diploma,CS,Trainee,R,"Flask, PyTorch, Scikit-learn",Onsite,3+,ml engineer
986,Both,Beginner,Data Science,Diploma,Non-CS,Pursuing Degree,C,"Django, Flask, PyTorch",Full-time Job,1+,ml engineer
987,Both,Beginner,Frontend,Bachelors,CS,Intern,"C++, Java","Django, Node",Full-time Job,3+,backend developer
991,Both,Intermediate,Data Science,Diploma,CS,Job Seeker,"C++, R","Django, Node",Remote,1+,backend developer


In [22]:
train_model(model , df)

C:\Users\Siddharth\AppData\Local\Temp\ipykernel_15972\2870069504.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].apply(lambda x : x.strip().lower())
C:\Users\Siddharth\AppData\Local\Temp\ipykernel_15972\2870069504.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = le.transform(df[col])
C:\Users\Siddharth\AppData\Local\Temp\ipykernel_15972\2870069504.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer